# 7. 시스템·VAR·VECM

식 여러 개를 함께 추정하는 시스템 추정(SUR·3SLS·FIML, 식 사이 제약), 변수 여러 개의 동태를 한꺼번에 보는 VAR, 공적분이
있는 변수의 VECM을 배우고, 그 결과를 연립 모델에 넣어 풉니다.

## 7.1 시스템 추정

`klein_like`는 교과서 Klein 모형 I의 구조로 만든 가상 자료입니다(소비·투자·민간 임금 세 식과 항등식 셋, [자료 설명](../../docs/reference/datasets.md)).
세 식 모두 같은 기의 내생변수(이윤 `PR`, 총수요 `X`, 임금 `WP`)를 설명변수로 가지므로 식마다 OLS를 하면 치우칩니다.

In [ ]:
import equion as eqn

kw = eqn.datasets.load("klein_like")
print(eqn.datasets.describe("klein_like").splitlines()[3:7])

시스템은 식을 `append`로 넣고 추정법을 고릅니다. 식은 추정식 꼴(계수 번호는 차례로 `c(1)`부터)이나 `c(n)` 꼴로 씁니다.
`@inst …`만 있는 줄은 모든 식의 도구변수입니다.

In [ ]:
kl = kw.system("klein")
kl.append("CONS c PR PR(-1) WP+WG",
          "I c PR PR(-1) K(-1)",
          "WP c X X(-1) A",
          "@inst G T WG A PR(-1) K(-1) X(-1)")
truth = [16.4, 0.125, 0.163, 0.79, 28.2, -0.013, 0.756, -0.195, 1.8, 0.4, 0.181, 0.15]
import pandas as pd
table = {"참값": truth}
for method in ("ols", "tsls", "three_sls"):
    getattr(kl, method)()
    table[method] = kl.coefs.round(3).tolist()
print(pd.DataFrame(table, index=kl.coefs.index))

OLS는 이윤 `PR`의 계수(`c(2)`·`c(6)`)를 크게 잘못 잡고, 도구변수를 쓰는 2SLS·3SLS는 참값 가까이로 돌아옵니다. 3SLS는 식 사이
오차 상관까지 써서 2SLS보다 효율적입니다(이 자료의 오차는 상관이 없어 비슷하다). 결과표는 `eqn.show(kl)`.

**FIML은 여기서 쓰지 않습니다.** 시스템에는 항등식을 넣을 수 없어 FIML은 시스템 식의 목표변수(`CONS`·`I`·`WP`)만 내생으로 보고,
항등식이 정하는 `PR`·`X`는 외생으로 다룹니다 — 그러면 결과가 OLS와 비슷해집니다. Equion은 이럴 때 경고합니다.

In [ ]:
kl.fiml()
print(kl.coefs.round(3)[["c(2)", "c(6)"]].to_dict())     # OLS 쪽으로 치우친다

FIML은 모든 같은 기 내생변수가 시스템 안 식의 목표변수인 모형(또는 항등식을 대입해 다시 쓴 모형)에서 쓰세요.

In [ ]:
kl.three_sls()
eqn.show(kl.equation_stats())
print(kl.sigma.round(3))                     # 식 사이 잔차 공분산

## 7.2 식 사이 제약

같은 `c(n)`을 두 식에 쓰면 그 계수를 같게 묶습니다. 또는 `restrict=["…"]`로 선형 제약을 줍니다. 제약은 Wald 검정으로 시험합니다.

In [ ]:
kl.three_sls()
kl.wald("c(3) = c(7)")                       # 소비·투자의 지난 이윤 계수가 같은가(참값 0.163·0.756 — 기각)
kl.three_sls(restrict=["c(3) = c(7)"])
print(kl.coefs.round(3)[["c(3)", "c(7)"]].to_dict())

## 7.3 시스템을 모델에 넣기

추정한 시스템은 계수를 고정한 방정식으로 모델에 들어갑니다. 항등식 셋을 더해 과거를 풀어 봅니다.

In [ ]:
kl.three_sls()
km = kw.model("klein_model").merge(kl)
km.append("X = CONS + I + G", "PR = X - T - WP", "K = K(-1) + I")
r = km.solve("1925", "1990")
eqn.show(r.summary())
print(km.stability("1960"))

순투자 `I`는 0 근처를 오가므로 RMSPE가 매우 크게 나옵니다 — 이런 계열은 MAE(수준)로 봅니다.
최대 근이 1을 조금 넘습니다 — 추정한 모형은 1960년 근처에서 아주 천천히 발산하는 동태를 가집니다(참 모형에서 크게 벗어나지
않은 추정 오차가 안정 경계를 넘긴 것). Klein 모형은 실제 자료에서도 안정 경계 근처로 알려져 있습니다. 근이 어떤 변수에
실렸는지는 `eqn.show(km.stability("1960"))`의 꼬리 줄에 나옵니다.

## 7.4 VAR

변수 몇 개를 서로의 시차로 설명합니다. `toy_q`의 소득 증가율과 정책금리로 해 봅니다. 시리즈 이름 대신 식을 써도 됩니다.

In [ ]:
wf = eqn.datasets.load("toy_q")
eqn.show(wf.var("v_sel", ["dlog(Y)", "R"], lags=1).lag_order(6))     # 시차 고르기(같은 표본에서 정보기준)
v = wf.var("v1", ["dlog(Y)", "R"], lags=2)
eqn.show(v)

`*`가 정보기준이 고른 시차입니다(BIC·HQ는 1, AIC·FPE는 3 — 작은 표본에서는 BIC 쪽이 안전하고, 동태를 넉넉히 보려면
AIC 쪽). 여기서는 그 사이인 2를 씁니다. 추정한 VAR로 Granger 인과, 충격반응, 예측오차 분산분해, 예측, 안정성을 봅니다.

In [ ]:
v.granger("dlog(Y)", "R")                    # R의 시차가 dlog(Y)를 설명하는가
irf = v.irf(8)                               # 촐레스키(변수 순서대로) 1 표준편차 충격
print(irf.xs("R", axis=1, level="충격").iloc[:4].round(5))
fe = v.fevd(8)
print(fe["dlog(Y)"].iloc[[0, 3, 7]].round(3))
print(v.forecast(4).round(4))
print(v.stability())

금리 충격은 같은 분기에는 소득 증가율에 영향이 없고(변수 순서상 `dlog(Y)`가 먼저), 다음 분기에 투자를 거쳐 조금 낮춘 뒤
사라집니다. 분산분해로 보면 소득 증가율 변동의 대부분은 자기 충격에서 옵니다.

## 7.5 VECM

장기금리 `RL`과 정책금리 `R`은 공적분 관계입니다(`RL = 1 + R + u`). VECM은 차분의 VAR에 균형 오차 `β'y(-1)`를 더한 꼴입니다.
순위는 Johansen 검정으로 고릅니다([튜토리얼 3](../../docs/tutorial/03-diagnostics.md#38-공적분)).

In [ ]:
wf.smpl("1980Q1", "2019Q4")                  # RL이 있는 구간
k = wf.vecm("vecm_rl", ["RL", "R"], lags_diff=1, rank=1)
print(k.beta.round(3))                       # 공적분 벡터(첫 변수로 정규화): RL − 1.0·R
print(k.alpha.round(3))                      # 조정 속도
print(k.forecast(4).round(3))

`β`가 [1, −1]에 가까워 참 관계 `RL − R`을 찾았고, `RL` 식의 조정 속도 α가 음수라 균형에서 벗어나면 `RL`이 되돌아옵니다.

## 7.6 VAR·VECM을 모델에 넣기

VAR·VECM도 계수를 고정한 식으로 모델에 들어갑니다 — 시나리오·충격을 다른 식과 똑같이 쓸 수 있습니다.

In [ ]:
mv = wf.model("m_vecm").merge(k)
print(mv.endogenous)
r = mv.solve("2015Q1", "2019Q4", mode="static")
print(r.counts())

## 정리

| 하려는 일 | 코드 |
|---|---|
| 시스템 | `s = wf.system("s"); s.append(식…, "@inst …")` → `.ols()` `.sur(iterate=True)` `.tsls()` `.three_sls()` `.fiml()` |
| 제약·검정 | 같은 `c(n)` · `restrict=["c(3) = c(7)"]` · `s.wald("…")` |
| 결과 | `s.coefs` · `s.stderrs` · `s.equation_stats()` · `s.sigma` · `s.resid` |
| VAR | `wf.var(이름, 변수들, lags)` → `.lag_order(p)` `.granger(a, b)` `.irf(h)` `.fevd(h)` `.forecast(h)` `.stability()` |
| VECM | `wf.vecm(이름, 변수들, lags_diff, rank, deterministic)` → `.alpha` `.beta` `.gamma` `.forecast(h)` |
| 모델로 | `wf.model("m").merge(s)` · `.merge(v)` · `.merge(k)` |

다음: [8. 명령줄](../../docs/tutorial/08-cli.md). 레시피: [시스템·VAR 레시피](../../docs/howto/system-var.md).